# MCSR JSONL Visualization

Before deciding features and preprocessing data, we should make the data in each .json entry more digestible. Having world stats, match stats, and a general timeline should make it much easier to understand the data that we are given.

#### Data and Package Imports

In [1]:
import json
from pathlib import Path


#### All event flags are stored in milliseconds. It's much easier to understand minutes:seconds when the milliseconds get too high

In [7]:
def time_to_mmss(milliseconds):
    if not isinstance(milliseconds, (int, float)): #make sure the input is actually a number
        return "whatdatmean (NaN)"

    total_seconds = int(milliseconds // 1000)

    minutes = total_seconds // 60
    seconds = total_seconds % 60

    return f"{minutes:02d}:{seconds:02d}"

#### Print general match info and a timeline for each player

In [22]:
def print_match_summary(jsonl_file, line_number):

    """load and parse through the jsonl"""

    if line_number < 1: #care me, ts is 1-indexed (i.e. 1 is the first line, not zero)
        raise ValueError("line_number must be >= 1")

    jsonl_path = Path(jsonl_file) #for now, we're just using the sample, but part of this can be used to parse through our actual json later

    with jsonl_path.open("r", encoding="utf-8") as f:
        target_line = None

        for current_line_num, line in enumerate(f, start=1): #extract the line we want
            if current_line_num == line_number:
                target_line = line.strip()
                break

    if target_line is None: #do we have data? hope so
        raise IndexError(f"line_number {line_number} is out of range for {jsonl_file}")
    if not target_line:
        raise ValueError(f"line_number {line_number} is empty")

    """get specific match data"""

    match = json.loads(target_line)

    seed = match.get("seed", {})
    overworld_type = seed.get("overworld", "MISSING")
    nether_type = seed.get("nether", "MISSING")

    # if the overworld is a village, i just wanna get the biome of the village. idk if it matters but im curious
    variations = seed.get("variations", [])
    village_biome = None
    if overworld_type == "VILLAGE":
        for variation in variations:
            if isinstance(variation, str) and variation.startswith("biome:structure:"): #the first 'biome:structure' seems to be pretty indicative of the starting biome
                village_biome = variation.split("biome:structure:", 1)[1]
                break

    players = match.get("players", [])
    players_by_uuid = {p.get("uuid"): p for p in players if p.get("uuid")}

    winner_uuid = match.get("result", {}).get("uuid")
    winner_time_ms = match.get("result", {}).get("time")
    winner_time_mmss = time_to_mmss(winner_time_ms)

    winner_player = players_by_uuid.get(winner_uuid, {})
    winner_name = winner_player.get("nickname", "MISSING")

    timelines = sorted(match.get("timelines", []), key=lambda e: e.get("time", float("inf"))) #get the timelines list, or an empty one if not available. then sort events by time. i dont think any events are missing a time, but if they are, time will be set to inf and theyll still be displayed

    # we'll infer how the match was won since its not an explicit flag in the json.
    winner_timeline_events = [
        event.get("type") for event in timelines if event.get("uuid") == winner_uuid
    ]

    if "end.kill_dragon" in winner_timeline_events or "projectelo.timeline.dragon_death" in winner_timeline_events:
        win_type = "dragon kill"
    elif match.get("forfeited") or "projectelo.timeline.forfeit" in [event.get("type") for event in timelines]:
        win_type = "forfeit"
    else:
        win_type = "unknown"

    """print results and timeline"""

    print(f"JSONL Line: {line_number} | Match ID: {match.get('id', 'MISSING')}")

    if village_biome:
        print(f"Overworld Type: {overworld_type} ({village_biome})")
    else:
        print(f"Overworld Type: {overworld_type}")

    print(f"Nether Type: {nether_type}")
    print(f"Winner: {winner_name} (time: {winner_time_mmss}, win type: {win_type})")

    print("\nPlayers:")
    for p in players:
        p_uuid = p.get("uuid", "MISSING")
        username = p.get("nickname", "MISSING")
        rank = p.get("eloRank")
        rank_display = rank if rank is not None else "N/A" #lotta entries dont have rank, probably doing their placements. might wanna drop from final dataset
        winner_tag = " (WINNER)" if p_uuid == winner_uuid else ""
        print(f"- Username: {username} | UUID: {p_uuid} | Elo: {rank_display}{winner_tag}") #im actually not sure if this is the elo before or after the match ngl

    print("\nTimelines:")

    for p in players:
        p_uuid = p.get("uuid")
        username = p.get("nickname", "MISSING")
        print(f"\n  {username} ({p_uuid}):")

        player_events = [event for event in timelines if event.get("uuid") == p_uuid]

        if not player_events:
            print("    - No timeline events")
            continue

        for event in player_events:
            event_type = event.get("type", "MISSING_EVENT")
            event_time_ms = event.get("time")
            event_time_mmss = time_to_mmss(event_time_ms)
            print(f"    - {event_time_mmss} ({event_time_ms} ms): {event_type}")

In [23]:
print_match_summary("mcSample.jsonl", line_number=1)

JSONL Line: 1 | Match ID: 4547059
Overworld Type: VILLAGE (savanna)
Nether Type: BRIDGE
Winner: Indsaet (time: 34:38, win type: dragon kill)

Players:
- Username: Indsaet | UUID: 33d7e36107f24d93948fc3fd1845fdf5 | Elo: 6664 (WINNER)
- Username: akavry | UUID: 030090a488c848dcb629f4e620bcc444 | Elo: N/A

Timelines:

  Indsaet (33d7e36107f24d93948fc3fd1845fdf5):
    - 00:24 (24173 ms): story.root
    - 00:33 (33792 ms): story.smelt_iron
    - 00:33 (33792 ms): story.mine_diamond
    - 00:33 (33792 ms): story.obtain_armor
    - 01:02 (62647 ms): adventure.root
    - 02:12 (132902 ms): story.mine_stone
    - 04:10 (250740 ms): husbandry.root
    - 04:38 (278535 ms): story.lava_bucket
    - 04:55 (295888 ms): story.enter_the_nether
    - 04:55 (295888 ms): nether.root
    - 05:57 (357948 ms): nether.find_bastion
    - 07:32 (452093 ms): story.form_obsidian
    - 07:59 (479460 ms): nether.obtain_crying_obsidian
    - 08:21 (501047 ms): nether.loot_bastion
    - 11:43 (703943 ms): nether.find

In [24]:
print_match_summary("mcSample.jsonl", line_number=2)

JSONL Line: 2 | Match ID: 4547060
Overworld Type: VILLAGE (desert)
Nether Type: TREASURE
Winner: boxbalex (time: 18:26, win type: forfeit)

Players:
- Username: boxbalex | UUID: b0604589651f4c44aad9e1894e8e51a8 | Elo: N/A (WINNER)
- Username: qurd145larry | UUID: a017203898ae43ae8d9d8b61dd188353 | Elo: N/A

Timelines:

  boxbalex (b0604589651f4c44aad9e1894e8e51a8):
    - 01:00 (60408 ms): story.iron_tools
    - 01:00 (60433 ms): story.form_obsidian
    - 01:00 (60433 ms): story.smelt_iron
    - 01:36 (96968 ms): adventure.root
    - 02:26 (146795 ms): story.root
    - 02:55 (175746 ms): story.mine_stone
    - 03:09 (189448 ms): story.lava_bucket
    - 03:11 (191292 ms): husbandry.root
    - 05:07 (307930 ms): story.enter_the_nether
    - 05:07 (307973 ms): nether.root
    - 06:04 (364207 ms): nether.find_bastion
    - 06:17 (377698 ms): nether.distract_piglin
    - 07:51 (471714 ms): projectelo.timeline.death
    - 08:06 (486516 ms): nether.obtain_crying_obsidian
    - 08:54 (534431 ms